# **R/S benchmark — PCE training**

This notebook **only** fits and validates the PCE. It reads the `dataset_unique_train` / `dataset_unique_val` files written by [`01_generate_dataset.ipynb`](01_generate_dataset.ipynb). Diagnostic plots (speed-up, KL-divergence check) are in [`02_plot_pce_validation.ipynb`](02_plot_pce_validation.ipynb).

## **1. Libraries**

In [ ]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import dill
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from functions import *
from UQpy.distributions import Normal, JointIndependent

/home/casa-wand/Documentos/2024-1_victor_hugo_renata_maria/.venv/lib/python3.11/site-packages/UQpy/__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## **2. Random variables and fixed parameters**

Must match [`01_generate_dataset.ipynb`](01_generate_dataset.ipynb). This only rebuilds the distribution object, it draws no new samples.

In [2]:
r_mean = 5.0
r_std  = 0.8
s_mean = 2.0
s_std  = 0.6

n_latent_samples = 2500   # must match stage 1 — it is the filename prefix
n_lambdas        = 4
max_degree       = 3        # maximum total degree of the PCE polynomial basis

r_dist = Normal(loc=r_mean, scale=r_std)
s_dist = Normal(loc=s_mean, scale=s_std)
joint  = JointIndependent(marginals=[r_dist, s_dist])

## **3. Time grid**

Must match times written by [`01_generate_dataset.ipynb`](01_generate_dataset.ipynb).

In [3]:
times = np.linspace(0, 150, 10, endpoint=True)
times

array([  0.        ,  16.66666667,  33.33333333,  50.        ,
        66.66666667,  83.33333333, 100.        , 116.66666667,
       133.33333333, 150.        ])

## **4. Load the datasets and train the PCE at each time step**

In [4]:
print("="*60)
print("TRAINING THE BENCHMARK PCE")
print("="*60)

results = []
for t in times:
    with open(f'{n_latent_samples}_dataset_unique_train_{t}_benchmark.pkl', 'rb') as f:
        df_unique_train = dill.load(f)
    with open(f'{n_latent_samples}_dataset_unique_val_{t}_benchmark.pkl', 'rb') as f:
        df_unique_val = dill.load(f)

    result = train_and_validate_pce_from_dataset_benchmark(
                                                              df_unique_train=df_unique_train,
                                                              df_unique_val=df_unique_val,
                                                              joint=joint,
                                                              time_step=t,
                                                              n_latent_samples=n_latent_samples,
                                                              n_lambdas=n_lambdas,
                                                              max_degree=max_degree,
                                                              output_dir='.',
                                                          )
    result['x_train'] = df_unique_train[['r', 's']].to_numpy()
    results.append(result)

TRAINING THE BENCHMARK PCE

----------------------------------------
TRAINING PCE FOR TIME STEP: 0.0 years
----------------------------------------
1. PCE training dataset has been saved!
2. PCE statistcs has been saved!

----------------------------------------
TRAINING PCE FOR TIME STEP: 16.666666666666668 years
----------------------------------------
1. PCE training dataset has been saved!
2. PCE statistcs has been saved!

----------------------------------------
TRAINING PCE FOR TIME STEP: 33.333333333333336 years
----------------------------------------
1. PCE training dataset has been saved!
2. PCE statistcs has been saved!

----------------------------------------
TRAINING PCE FOR TIME STEP: 50.0 years
----------------------------------------
1. PCE training dataset has been saved!
2. PCE statistcs has been saved!

----------------------------------------
TRAINING PCE FOR TIME STEP: 66.66666666666667 years
----------------------------------------


1. PCE training dataset has been saved!
2. PCE statistcs has been saved!

----------------------------------------
TRAINING PCE FOR TIME STEP: 83.33333333333334 years
----------------------------------------
1. PCE training dataset has been saved!
2. PCE statistcs has been saved!

----------------------------------------
TRAINING PCE FOR TIME STEP: 100.0 years
----------------------------------------
1. PCE training dataset has been saved!
2. PCE statistcs has been saved!

----------------------------------------
TRAINING PCE FOR TIME STEP: 116.66666666666667 years
----------------------------------------
1. PCE training dataset has been saved!
2. PCE statistcs has been saved!

----------------------------------------
TRAINING PCE FOR TIME STEP: 133.33333333333334 years
----------------------------------------
1. PCE training dataset has been saved!
2. PCE statistcs has been saved!

----------------------------------------
TRAINING PCE FOR TIME STEP: 150.0 years
-----------------------

1. PCE training dataset has been saved!
2. PCE statistcs has been saved!


## **5. Validation summary**

How well the PCE reproduces each lambda, per time step.

In [5]:
validation_summary = pd.concat([r['statistics'] for r in results], ignore_index=True)
validation_summary.insert(0, 'Time (years)', [r['time_step'] for r in results])
validation_summary

,Time (years),MSE λ1,MSE λ2,MSE λ3,MSE λ4,R² λ1,R² λ2,R² λ3,R² λ4
0,0.000000,0.000028,0.061789,0.000428,0.000412,0.999971,0.965079,0.006096,0.010910
1,16.666667,0.000026,2.457809,0.001599,0.001524,0.999969,0.509918,-0.047663,-0.002007
2,33.333333,0.000030,0.064444,0.000476,0.000460,0.999959,0.977622,0.003670,-0.020467
3,50.000000,0.000025,11.880267,0.005019,0.004772,0.999959,0.321168,0.007467,0.025255
4,66.666667,0.000024,6.963671,0.003861,0.003895,0.999954,0.420201,0.006426,0.000134
5,83.333333,0.000022,6.137892,0.001696,0.001498,0.999951,0.542170,-0.032421,-0.014089
6,100.000000,0.000022,8.966568,0.002930,0.003188,0.999947,0.396960,-0.102912,-0.098744
7,116.666667,0.000021,15.206179,0.004861,0.004889,0.999944,0.397579,0.012168,0.013320
8,133.333333,0.000019,7.150622,0.002914,0.002874,0.999947,0.442911,-0.048578,-0.058713
9,150.000000,0.000023,6.949432,0.001623,0.001596,0.999936,0.508792,-0.032723,-0.021916
